# Problem Set 2: the W mass and the Jacobian peak — version 2

Before editing, copy this template from
`course-materials/notebooks/templates/02_w_transverse_mass_v2.ipynb`
into your `work` folder in JupyterLab (create the folder if needed).
Open and complete that copy so course syncs do not interfere with your work.
Do not copy the event files: the notebook finds them under the FindingZ run root.

Generate **1,000 events per sample** in FindingZ, using the Standard Model,
LHC Run 2 and **MadGraph-only** mode. No Pythia or Delphes is needed.
This notebook reads the saved `.lhe` or `.lhe.gz` event files directly;
it does not require changes to FindingZ.

Use these process definitions in the generation card (or the corresponding course presets):

**W sample**
```
generate p p > w+, w+ > e+ ve
add process p p > w-, w- > e- ve~
```
**W + jet sample**
```
generate p p > w+ j, w+ > e+ ve
add process p p > w- j, w- > e- ve~
```
For the second sample set `ptj = 30` GeV in the run card. Keep other cuts the same;
record the actual lepton cuts, W mass and width in your saved cards. Use fresh runs.
Here `j` is an outgoing parton, not a reconstructed detector jet.

First do the short derivation in the problem set. Then complete the short analysis cells below,
make the plots and interpret them. The file-loading and plotting code is supplied.
The neutrino's full momentum is used only to check the true invariant mass;
the transverse-mass calculation must use transverse components alone.

## How to load your runs
1. Run the next code cell with **Shift+Enter**. Leave its settings unchanged first: it reads the run-root setting automatically and prints the directory it is using.
2. The output lists your completed samples as `Run ID | process | created | events | label`. `w_enu` is W without a jet; `wj_enu` is W + jet.
3. In the following code cell, paste the matching IDs **between the quotes** in `W_RUN_ID` and `WJ_RUN_ID`. For example, if the list shows an ID `abc123` for your W sample, enter `W_RUN_ID = "abc123"`. Use the actual IDs from your list, not this example. If there is only one sample of each type, leave both strings empty.
4. Run that cell with **Shift+Enter**. It finds and loads the event files for those runs. No individual event-file paths are needed.

**Only if the printed run-root directory is wrong:** enter the course-provided directory in `RUN_ROOT_OVERRIDE` below, for example `RUN_ROOT_OVERRIDE = "/home/jovyan/findingz-runs"`, and rerun the cell. This must be the directory containing all run folders, not one run folder or an `.lhe` file. The example is the course configuration's default; if it does not exist and no environment setting is available, ask the instructor for the server's run-root path.


In [ ]:
from pathlib import Path
import os, gzip, json
import xml.etree.ElementTree as ET
import numpy as np
import matplotlib.pyplot as plt

MW = 80.4  # GeV: replace with MASS 24 in your saved param_card.dat
# Usually leave this empty. Paste the course-provided run-root path here only if needed.
RUN_ROOT_OVERRIDE = ""
RUN_ROOT = Path(RUN_ROOT_OVERRIDE or os.environ.get('FINDINGZ_RUN_ROOT', '~/findingz-runs')).expanduser()
print('FindingZ run root:', RUN_ROOT)

def discover_runs(root):
    if not root.is_dir():
        raise FileNotFoundError(f'{root} is not a directory. Enter the course-provided path in RUN_ROOT_OVERRIDE above, then rerun this cell.')
    runs = {}
    manifests = sorted(root.glob('*/manifest.json'))
    print(f'Found {len(manifests)} run manifests in {root}')
    for manifest_path in manifests:
        try:
            manifest = json.loads(manifest_path.read_text())
        except (OSError, ValueError) as exc:
            print('Skipping unreadable manifest:', manifest_path, exc)
            continue
        config = manifest.get('config', {})
        if manifest.get('status') != 'complete':
            print('Not completed:', manifest_path.parent.name, 'status:', manifest.get('status'))
            continue
        relative = manifest.get('stages', {}).get('matrix_element')
        if not relative:
            print('No saved LHE checkpoint:', manifest_path.parent.name, 'process:', config.get('process'))
            continue
        path = manifest_path.parent / relative
        if not path.is_file():
            print('Missing LHE file for run:', manifest_path.parent.name)
            continue
        runs[manifest_path.parent.name] = dict(
            path=path, process=config.get('process', 'unknown'), label=manifest.get('label', ''),
            created=manifest.get('created_at', ''), events=config.get('n_events', config.get('events', '?')))
    return runs

runs = discover_runs(RUN_ROOT)
print('Run ID | process | created | events | label')
for run_id, run in runs.items():
    print(run_id, run['process'], run['created'], run['events'], run['label'], sep=' | ')
if not runs:
    print('No completed runs with saved LHE files found. Check the run-root and diagnostic lines above; this does not necessarily mean no runs were generated.')

## Load the two samples
The cell above uses FindingZ's run root and lists all completed runs with saved LHE files, including runs made with custom process names.
If there is exactly one run for each course preset, the next cell selects them automatically.
If you have several, enter their **run IDs** from the list in `W_RUN_ID` and
`WJ_RUN_ID`. You do not need to locate or type event-file paths: the notebook
reads them from each run's manifest.
Only numpy and matplotlib are needed beyond Python's standard library.
The loader checks for one electron/positron and one electron neutrino/antineutrino
per event and equal positive event weights (the unweighted LO samples requested here).

In [ ]:
W_RUN_ID = ""   # Leave blank if exactly one W run is listed.
WJ_RUN_ID = ""  # Otherwise copy the desired run ID from the list above.

def select_run(runs, run_id, process):
    candidates = [key for key, value in runs.items() if value['process'] == process]
    if not run_id:
        if len(candidates) != 1:
            raise ValueError(f'Choose a run ID for {process}; available IDs: {candidates}')
        run_id = candidates[0]
    if run_id not in runs:
        raise ValueError(f'{run_id!r} is not a completed run with saved LHE files in the list above.')
    if runs[run_id]['process'] != process:
        print('Different process ID:', runs[run_id]['process'],
              '-- check that this is your intended W or W+jet sample.')
    print('Selected:', run_id)
    return runs[run_id]['path']

W_FILE = select_run(runs, W_RUN_ID, 'w_enu')
WJ_FILE = select_run(runs, WJ_RUN_ID, 'wj_enu')

def read_sample(path):
    electrons, neutrinos, jet_pts, weights = [], [], [], []
    opener = gzip.open if path.suffix == '.gz' else open
    with opener(path, 'rt') as stream:
        for _, element in ET.iterparse(stream, events=('end',)):
            if element.tag != 'event':
                continue
            lines = [line.strip() for line in (element.text or '').splitlines()
                     if line.strip() and not line.lstrip().startswith('#')]
            header = lines[0].split()
            n = int(header[0])
            number = lambda x: float(x.replace('D', 'E').replace('d', 'e'))
            weights.append(number(header[2]))
            final = []
            for line in lines[1:n+1]:
                fields = line.split()
                if int(fields[1]) == 1:
                    # Four-vector order is (E, px, py, pz).
                    final.append((int(fields[0]), np.array([
                        number(fields[9]), number(fields[6]),
                        number(fields[7]), number(fields[8])])))
            e = [p for pid, p in final if abs(pid) == 11]
            nu = [p for pid, p in final if abs(pid) == 12]
            if len(e) != 1 or len(nu) != 1:
                raise ValueError('Choose a W -> e nu sample with exactly one e and one nu per event.')
            electrons.append(e[0]); neutrinos.append(nu[0])
            jet_pts.append(max([np.hypot(p[1], p[2]) for pid, p in final
                                if abs(pid) in {1,2,3,4,5,21}], default=0.))
            element.clear()
    if not electrons:
        raise ValueError(f'No events found in {path}')
    weights = np.array(weights)
    if not np.all(weights > 0) or not np.allclose(weights / weights[0], 1.):
        raise ValueError('Use equal-positive-weight LO events; these plots do not implement reweighting.')
    e, nu = np.array(electrons), np.array(neutrinos)
    sample = {'electron': e, 'neutrino': nu, 'jet_pt': np.array(jet_pts)}
    print(path.name, ':', len(e), 'events')
    if len(e) != 1000:
        print('Expected 1,000 events: check the generation settings.')
    return sample

samples = {'W': read_sample(W_FILE), 'W + jet': read_sample(WJ_FILE)}


## Your analysis
The loading code above is supplied; you do not need to understand the LHE format.
Each row of `electron` or `neutrino` is one event, with columns **E, px, py, pz** in GeV.
Complete the lines marked `TODO`. A `None` is a placeholder: replace it with an expression.
Use `np.sqrt`, `np.arctan2(py, px)` and `np.cos` to operate on all events at once.

### 1. W sample: construct the observables
Calculate the electron and neutrino transverse momenta and azimuths, then calculate
transverse mass using your result from the problem set. Finally calculate the true
invariant mass from the *sum* of the electron and neutrino four-momenta.
Only the truth calculation may use the neutrino's longitudinal momentum.

In [ ]:
electron = samples['W']['electron']
neutrino = samples['W']['neutrino']

E_e, px_e, py_e, pz_e = electron.T
E_nu, px_nu, py_nu, pz_nu = neutrino.T

pt_e = None       # TODO: electron transverse momentum
pt_nu = None      # TODO: neutrino transverse momentum
phi_e = np.arctan2(py_e, px_e)
phi_nu = np.arctan2(py_nu, px_nu)
mt = None         # TODO: transverse mass

E = E_e + E_nu
px = px_e + px_nu
py = py_e + py_nu
pz = pz_e + pz_nu
mass = None       # TODO: invariant mass; use np.maximum(..., 0) before taking a square root
recoil = None     # TODO: transverse momentum of the electron + neutrino pair

assert all(x is not None for x in [pt_e, pt_nu, mt, mass, recoil]), 'Complete the TODO lines.'
print('Events loaded:', len(pt_e))
print('Largest W transverse momentum [GeV]:', np.max(recoil))

### 2. Make the mass-estimator plot
The plotting commands below are supplied as a simple example. All three curves use
the same 2 GeV bins. We divide counts by the total number of events and bin width;
events outside the displayed range therefore do not change the normalization.

Estimate the W mass from the plot. Explain why the blue and orange curves should
coincide for this sample. Check that numerically in the next cell.

In [ ]:
bins = np.arange(0, 142, 2)
weights = np.ones(len(pt_e)) / (len(pt_e) * 2)

plt.figure(figsize=(8, 4.5))
plt.hist(2*pt_e, bins=bins, weights=weights, histtype='step',
         color='tab:blue', linewidth=4, label=r'$2p_{Te}$')
plt.hist(mt, bins=bins, weights=weights, histtype='step',
         color='tab:orange', linewidth=2, linestyle='--', label=r'$m_T$')
plt.hist(mass, bins=bins, weights=weights, histtype='step',
         color='tab:green', linewidth=2, linestyle=':', label=r'$m_{e\nu}$')
plt.axvline(MW, color='grey', linestyle='-.', linewidth=1)
plt.xlim(0, 140)
plt.xlabel('Mass estimator [GeV]')
plt.ylabel('Probability density [1/GeV]')
plt.title('W without an additional jet')
plt.legend()
plt.show()

In [ ]:
difference = None  # TODO: largest absolute difference between mt and 2*pt_e
assert difference is not None, 'Complete the numerical comparison.'
print('Largest difference [GeV]:', difference)

**Your answer:** What mass do you infer? Why do the two estimators coincide?

*Write two or three sentences here.*

### 3. Compare with the Jacobian prediction
In your written work, derive the distribution for isotropic decay.
Here we sample that same reference distribution: `cos_theta` is uniform from -1 to 1.
Use $p_T=(m_W/2)\sin\theta_*$ to calculate its transverse momenta.
These reference events have a fixed mass and no cuts; they are not the generated W sample.

In [ ]:
rng = np.random.default_rng(225)
cos_theta = rng.uniform(-1, 1, 100000)
pt_reference = None  # TODO: use cos_theta and MW to calculate pT for the isotropic reference
assert pt_reference is not None, 'Complete the reference pT formula.'

bins_pt = np.arange(0, 62, 2)
plt.figure(figsize=(8, 4.5))
plt.hist(pt_e, bins=bins_pt, weights=np.ones(len(pt_e))/(len(pt_e)*2),
         histtype='step', linewidth=3, label='Generated W sample')
plt.hist(pt_reference, bins=bins_pt,
         weights=np.ones(len(pt_reference))/(len(pt_reference)*2),
         histtype='step', linewidth=2, linestyle='--', label='Isotropic reference')
plt.axvline(MW/2, color='grey', linestyle=':')
plt.xlim(0, 60)
plt.xlabel('Electron transverse momentum [GeV]')
plt.ylabel('Probability density [1/GeV]')
plt.legend()
plt.show()

**Your answer:** Explain the peak using the Jacobian you derived. Do the generated
events follow the isotropic reference exactly? Why need they not?
Record the generation cuts when making this comparison.

*Write two or three sentences here.*

### 4. Add transverse recoil
Repeat your calculation for the W + jet sample below. Use new names ending in `_j`
so you can still compare with the first sample. Copy the relevant formulas from
section 1, changing the input arrays. Check that the outgoing parton passes the
requested 30 GeV cut; if it does not, check the generation settings.

In [ ]:
electron_j = samples['W + jet']['electron']
neutrino_j = samples['W + jet']['neutrino']
print('Smallest parton pT [GeV]:', samples['W + jet']['jet_pt'].min())

pt_e_j = None    # TODO
mt_j = None      # TODO
mass_j = None    # TODO
recoil_j = None  # TODO
assert all(x is not None for x in [pt_e_j, mt_j, mass_j, recoil_j]), 'Complete the W + jet calculation.'

Copy the plotting cell from section 2 into the empty cell below. Replace `pt_e`,
`mt`, `mass` by `pt_e_j`, `mt_j`, `mass_j`, and update the title and weights.
Keep the same bins, axis limits and line styles so the plots can be compared.

In [ ]:
# Your W + jet plot here.

### 5. Test the bounds on the actual events
Count how many W + jet events have $2p_{Te}>m_{e\nu}$ and how many have
$m_T>m_{e\nu}$. Use `np.count_nonzero(condition)` and the small tolerance below
to avoid counting floating-point roundoff. Divide by the number of events to get fractions.
Compare with the **actual invariant mass**, not the nominal mass parameter.

In [ ]:
tolerance = 1e-6 * np.maximum(mass_j, 1.)
n_above_2pt = None  # TODO: count 2*pt_e_j > mass_j + tolerance
n_above_mt = None   # TODO: count mt_j > mass_j + tolerance
assert n_above_2pt is not None and n_above_mt is not None, 'Complete the event counts.'
print('Fraction above true mass using 2pT:', n_above_2pt / len(mass_j))
print('Fraction above true mass using mT:', n_above_mt / len(mass_j))
print('Events with 2pT outside the 0--140 GeV plot:', np.count_nonzero(2*pt_e_j > 140))

**Your conclusion:** Which mass bound survives the transverse recoil? Explain why
using the kinematics, and support your answer with your plots and event fractions.
Why is this useful when the neutrino's longitudinal momentum cannot be measured?

*Write your answer here.*